# 06. Нейросетевая модель текста: rubert-tiny2

Финальная модель читает текст через TF-IDF: она видит, **какие слова** встречаются, но не их смысл.
Нейросетевая языковая модель переводит текст объявления в вектор, в котором объявления
с похожим смыслом оказываются рядом, даже если слова разные («повар» и «кулинар»).

**Модель:** [`cointegrated/rubert-tiny2`](https://huggingface.co/cointegrated/rubert-tiny2), небольшой
BERT для русского языка (≈ 29 млн параметров, вектор из 312 чисел). Работает на обычном ноутбуке без GPU.

**Сравниваемые варианты:**

| Вариант | Что это |
|---|---|
| Ridge на TF-IDF | финальная модель из ноутбука 03 (точка отсчёта) |
| Ridge на эмбеддингах | только векторы нейросети + категории и числа |
| Нейросеть (MLP) на эмбеддингах | двухслойная нейросеть поверх векторов |
| Ridge на TF-IDF + эмбеддингах | оба представления текста вместе |
| Смесь | среднее прогнозов TF-IDF-модели и модели на эмбеддингах |

Сравнение на той же 5-кратной групповой кросс-валидации и том же тесте.

> **Перед запуском:** `pip install sentence-transformers` (вместе с ним ставится PyTorch).
> При первом запуске модель скачивается из интернета (≈ 120 МБ). Векторы сохраняются в
> `data/embeddings_*.npy`, поэтому повторные запуски быстрые.

In [ ]:
import os, sys, json, time, warnings
sys.path.insert(0, "../src")
import prep

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
FIG = "../reports/figures/"
MODEL_NAME = os.environ.get("EMB_MODEL", "cointegrated/rubert-tiny2")

meta = json.load(open("../models/final_model_meta.json"))
p, COLS = meta["params"], meta["features"]
C, N, B = prep.CAT_FEATURES, prep.NUM_FEATURES, prep.BOOL_FEATURES

data = prep.labeled(prep.load())
print(f"Вакансий: {len(data)} | модель: {MODEL_NAME}")

## 1. Векторы текста (эмбеддинги)

Для каждой вакансии считаются два вектора: по **названию** и по **полному тексту**
(название + требования + обязанности, без чисел и валют, как и в TF-IDF-модели).

In [ ]:
from sentence_transformers import SentenceTransformer

def embed(texts, tag):
    path = f"../data/embeddings_{tag}_{MODEL_NAME.split('/')[-1]}.npy"
    ids_path = path.replace(".npy", "_ids.npy")
    if os.path.exists(path) and os.path.exists(ids_path) and np.array_equal(np.load(ids_path), data["id"].values):
        return np.load(path)
    model = SentenceTransformer(MODEL_NAME)
    model.max_seq_length = 512
    t0 = time.time()
    E = model.encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    print(f"{tag}: {E.shape} за {time.time() - t0:.0f} с")
    np.save(path, E); np.save(ids_path, data["id"].values)
    return E

E_title = embed(data["title"].tolist(), "title")
E_full = embed(data["full"].tolist(), "full")
E = np.hstack([E_title, E_full])
print("Размер вектора на вакансию:", E.shape[1])

In [ ]:
data["pos"] = np.arange(len(data))          # строка вектора для каждой вакансии
train, test = prep.split(data)
E_tr, E_te = E[train["pos"].values], E[test["pos"].values]
folds = prep.cv_folds(train)
print(f"Обучение: {len(train)} | тест: {len(test)}")

## 2. Модели

In [ ]:
def tfidf_features():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=p["min_freq"]), C),
        ("num", StandardScaler(), N + B),
        ("t", TfidfVectorizer(min_df=p["min_df"], sublinear_tf=True, ngram_range=(1, p["word_ngram"]),
                              max_features=p["word_max"]), "full"),
        ("ti", TfidfVectorizer(min_df=p["min_df"], sublinear_tf=True, analyzer="char_wb",
                               ngram_range=(2, p["char_max"])), "title"),
    ], transformer_weights={"ti": p["title_weight"]})

def tab_features():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=p["min_freq"]), C),
        ("num", StandardScaler(), N + B),
    ])

class TfidfRidge:
    def fit(self, d, Ed): self.m = make_pipeline(tfidf_features(), Ridge(alpha=p["alpha"])).fit(d[COLS], d.y); return self
    def predict(self, d, Ed): return self.m.predict(d[COLS])

class EmbModel:
    """Категории и числа + векторы нейросети; регрессор на выбор."""
    def __init__(self, reg, emb_weight=1.0, dense=False):
        self.reg, self.w, self.dense = reg, emb_weight, dense
    def _X(self, d, Ed, fit):
        T = self.tab.fit_transform(d) if fit else self.tab.transform(d)
        X = sp.hstack([T, sp.csr_matrix(Ed * self.w)]).tocsr()
        return X.toarray() if self.dense else X
    def fit(self, d, Ed):
        self.tab = tab_features(); self.reg.fit(self._X(d, Ed, True), d.y); return self
    def predict(self, d, Ed): return self.reg.predict(self._X(d, Ed, False))

class TfidfPlusEmb:
    def __init__(self, emb_weight=1.0): self.w = emb_weight
    def fit(self, d, Ed):
        self.f = tfidf_features(); X = sp.hstack([self.f.fit_transform(d[COLS]), sp.csr_matrix(Ed * self.w)])
        self.m = Ridge(alpha=p["alpha"]).fit(X.tocsr(), d.y); return self
    def predict(self, d, Ed):
        return self.m.predict(sp.hstack([self.f.transform(d[COLS]), sp.csr_matrix(Ed * self.w)]).tocsr())

class Blend:
    def __init__(self, makers, weights): self.makers, self.weights = makers, weights
    def fit(self, d, Ed): self.ms = [mk().fit(d, Ed) for mk in self.makers]; return self
    def predict(self, d, Ed): return sum(w * m.predict(d, Ed) for w, m in zip(self.weights, self.ms))

MODELS = {
    "Ridge на TF-IDF (финальная)": lambda: TfidfRidge(),
    "Ridge на эмбеддингах": lambda: EmbModel(Ridge(alpha=1.0), emb_weight=3.0),
    "Нейросеть (MLP) на эмбеддингах": lambda: EmbModel(MLPRegressor(hidden_layer_sizes=(256, 64), alpha=1e-3,
                                                                   learning_rate_init=1e-3, early_stopping=True,
                                                                   max_iter=300, random_state=42), dense=True),
    "Ridge на TF-IDF + эмбеддингах": lambda: TfidfPlusEmb(emb_weight=1.0),
    "Смесь TF-IDF и эмбеддингов": lambda: Blend([lambda: TfidfRidge(),
                                                 lambda: EmbModel(Ridge(alpha=1.0), emb_weight=3.0)], [0.6, 0.4]),
}

## 3. Кросс-валидация

In [ ]:
rows = []
for name, make in MODELS.items():
    t0 = time.time()
    for k, (a, b) in enumerate(folds):
        m = make().fit(train.iloc[a], E_tr[a])
        rows.append({"модель": name, "фолд": k, **prep.metrics(train.iloc[b].y.values, m.predict(train.iloc[b], E_tr[b]))})
    print(f"{name:34s} {time.time() - t0:5.0f} с", flush=True)
cv = pd.DataFrame(rows)
summary = cv.groupby("модель").agg(MAE=("MAE, сом", "mean"), MAE_std=("MAE, сом", "std"),
                                   медианная=("медианная ошибка, сом", "mean"), R2=("R² (log)", "mean")).sort_values("MAE")
summary.style.format({"MAE": "{:,.0f}", "MAE_std": "±{:,.0f}", "медианная": "{:,.0f}", "R2": "{:.3f}"})

## 4. Тест

In [ ]:
rows = []
for name, make in MODELS.items():
    m = make().fit(train, E_tr)
    rows.append({"модель": name, **prep.metrics(test.y.values, m.predict(test, E_te))})
res = pd.DataFrame(rows).set_index("модель").sort_values("MAE, сом")
res.style.format({"MAE, сом": "{:,.0f}", "медианная ошибка, сом": "{:,.0f}", "MAPE, %": "{:.1f}",
                  "доля ошибок < 20%": "{:.0%}", "R² (log)": "{:.3f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
r = res["MAE, сом"][::-1]
ax.barh(r.index, r.values, color=["#2e9e5b" if "финальная" in n else "#3b7dd8" for n in r.index])
for i, v in enumerate(r.values): ax.text(v, i, f"  {v:,.0f}", va="center")
ax.set_xlim(0, r.max() * 1.2); ax.set_xlabel("MAE на тесте, сом")
ax.set_title("Классическая модель на TF-IDF против нейросетевых эмбеддингов")
plt.tight_layout(); plt.savefig(FIG + "06_neural.png", dpi=150); plt.show()

## Выводы

> Ноутбук запускается локально (модель скачивается из интернета). После запуска допишите выводы
> по таблицам выше, ответив на три вопроса:
> 1. Лучше ли эмбеддинги нейросети, чем TF-IDF, **сами по себе**?
> 2. Даёт ли **сочетание** TF-IDF и эмбеддингов выигрыш, и больше ли он разброса по фолдам (`MAE_std`)?
> 3. Стоит ли выигрыш усложнения (размер модели, время, зависимость от PyTorch)?